# Datamine IJKGEN Process Example

This notebook demonstrates how to configure and run the **`ijkgen`** process wrapper in `dmstudio`.

## Process Description

## IJKGEN

Recalculates or calculates the IJK field for a model.

To recalculate IJK values for an existing model, this model is entered on the &**IN** file, and a prototype model (as produced by [PROTOM](<protom.md>)) entered as the &**PROTO** file. The output file &**OUT** may be the same as the input, if required. Parameter @**PSMODEL** is set to zero in this case, meaning additional attributes other than IJK will not be transferred to the output file.

@**PSMODEL** =1 can be used to append the output file with the model fields of the input file, overwriting existing fields as required.

Recalculation of IJK values is often carried out to expand a model (for example, when the original model defined did not cover sufficient volume to include all the waste rock required for an open pit). For this, it is only necessary to define the new origin, number of cells etc. using [PROTOM](<protom.md>) before using **IJKGEN**.

#### Building a model from cell centre values

To create a valid model file from a file containing just cell centre co-ordinates, the output file must be a different name from the input. Parameter @**PSMODEL** is set to 1 to ensure that all model fields are written across from the prototype to the output file. The input file does not have to be sorted.

#### Building a model from a regular set of values

Block models coming into your application from other systems often comprise a set of values on a regular grid, with no positional data. Such data can be turned into a Datamine cell model with the aid of [EXTRA](<extra.md>) and IJKGEN. The stages are:-

1\. Enter the data as a Datamine file, with one record per block.

2\. Use [EXTRA](<extra.md>) to compute cell centre co-ordinates for each block.

3\. Use IJKGEN as described above to generate a model.

4\. Sort the model on IJK for speed of access.

### The Prototype Model

The input prototype model defines the cell sizes, origin, model dimensions etc. that will appear in the output model. Thus the process [PROTOM](<protom.md>) should be used prior to **IJKGEN** to set up this prototype model. It is very important to ensure that the model origin, cell sizes etc. are chosen so that the bottom left hand corner of a cell is at the model origin. Cell sizes should be defined to be the same as in the input model on &IN, or you run the risk of creating a model with overlapping cells or gaps between cells.

The **IJKGEN** process does not check for this, as there are circumstances where this is useful (for example, creating a pseudo model from blast hole data, where each hole is assigned to a cell with unit volume).

Note that **IJKGEN** will include rotated model fields, if found in the model prototype, to the resulting output (sorted) file).

### Data lying outside the model

If the data on &IN lies outside the model limits as defined in the prototype, then the IJK field will be set to absent data with a warning message. You must remove or change such IJK values before entering processes.

### Maximum IJK Value

The maximum IJK value is 2,147,483,647. This means you can have a model with, for example, 1400x1400x1000 parent cells.

See [Block Model Size Limits](<../COMMON/Block_Models_Size_Limits.md>).

### Input Files:

* **proto** (Block Model Prototype):
  Input prototype model describing the model parameters. Normally set up by PROTOM. Must
  contain the numeric fields XC, YC, ZC, IJK (explicit) and XMORIG, YMORIG, ZMORIG, NX,
  NY, NZ (implicit) and XINC, YINC, ZINC (either, as required). For recalculation of IJK
  in an existing model, may be the same file as IN.
  Required=Yes

* **in** (Input file to be converted into a model. Must contain the fields X , Y and Z representing (sub-)cell centre locations. This can be an existing model for recalculation of IJK.):
  Overwritten
  Required=No

### Output Files:

* **out** (BlockModel):
  Output model file. May be the same as IN where IN already contains model fields; in
  this case, recalculation is in-place. IJK will be set to absent (-) if the record lies
  outside the model limits.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  Explicit numeric field in IN containing the X co-ordinate of the (sub-)cell centre.
  Default=X
  Required=Yes

* **y** (Numeric : IN):
  Explicit numeric field in IN containing the Y co-ordinate of the (sub-)cell centre.
  Default=Y
  Required=Yes

* **z** (Numeric : IN):
  Explicit numeric field in IN containing the Z co-ordinate of the (sub-)cell centre.
  Default=Z
  Required=Yes

### Parameters:

* **psmodel**:

* **Options** (0: Just generate IJK field without copying additional attributes to the output):
  model. This is the recommended option if your output file already contains the expected
  attributes.; 1: Place all other model fields as well as IJK into the output model. This
  will copy standard fields from the prototype to the output file, overwriting any fields
  of the same name copied from the input points file.
  Range=0,1
  Values=0,1
  Default=0
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('ijkgen')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute ijkgen
print("Running ijkgen...")
dm_cmd.ijkgen(
    proto_i='t_mod1',  # required
    in_i='t_assays',  # required
    out_o='t_ijkgen_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    psmodel_p=1,  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("ijkgen execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_ijkgen_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")